In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import torch
import pandas as pd
from torch.utils.data import DataLoader

from src.dataset import SatelliteDataset
from src.models import get_resnet18_feature_extractor


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [3]:
train_df = pd.read_csv("../data/raw/train.csv")

dataset = SatelliteDataset(
    train_df,
    "../data/images/train"
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


In [4]:
model = get_resnet18_feature_extractor().to(device)

all_embeddings = []

with torch.no_grad():
    for images in loader:
        images = images.to(device)
        features = model(images)
        all_embeddings.append(features.cpu())


d:\Satellite Imagery-Based Property Valuation(CDC)\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Satellite Imagery-Based Property Valuation(CDC)\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ROG/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:14<00:00, 3.23MB/s]


In [5]:
embeddings = torch.cat(all_embeddings).numpy()
embeddings.shape


(16209, 512)

In [6]:
import numpy as np

np.save("../data/train_image_embeddings.npy", embeddings)
